# Stages 0–2 Colab (portable)

This notebook **clones GitHub** and **points at datasets on Google Drive**. It runs Stage 1 first and offers the approved Stage 2 frozen-CLIP retrieval experiment afterward. It does **not** train CLIP, use GPT-2, generate captions, integrate a decoder, or start an extension.

Repo: `https://github.com/yakshini12/remote-sensing-rag` (private unless you change that).

Recommended Drive layout:
```
MyDrive/rsrag-data/ucm_captions/     # extracted UCM-Captions
MyDrive/rsrag-data/rsitmd/           # extracted RSITMD
MyDrive/rsrag-artifacts/processed/
MyDrive/rsrag-artifacts/reports/
MyDrive/rsrag-artifacts/stage2/
```
Keep original zip files on Drive. Prefer unzipping to `/content` if Drive has too many files.

In [ ]:
# 1) Mount Drive (your Google account, in the browser — never paste a password here)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Clone the CODE from GitHub into the Colab VM (ephemeral).
# If the repo is private, Colab will ask you to authenticate git, or use a token
# you create yourself. Do not commit tokens.
import os
os.chdir("/content")
if not os.path.isdir("/content/remote-sensing-rag/.git"):
    !git clone https://github.com/yakshini12/remote-sensing-rag.git
else:
    !git -C /content/remote-sensing-rag pull --ff-only
%cd /content/remote-sensing-rag
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 3) Portable path overrides — edit only if your Drive folders differ.
import os
os.environ["RSRAG_PROJECT_ROOT"] = "/content/remote-sensing-rag"
os.environ["RSRAG_UCM_ROOT"] = "/content/drive/MyDrive/rsrag-data/ucm_captions"
os.environ["RSRAG_RSITMD_ROOT"] = "/content/drive/MyDrive/rsrag-data/rsitmd"
os.environ["RSRAG_PROCESSED_DIR"] = "/content/drive/MyDrive/rsrag-artifacts/processed"
os.environ["RSRAG_REPORT_DIR"] = "/content/drive/MyDrive/rsrag-artifacts/reports"

!python scripts/run_stage0_stage1.py --config configs/stage0_stage1_ucm_rsitmd.yaml --print-paths

In [ ]:
# 4) Always-safe synthetic smoke test (no real datasets required)
!python scripts/make_fixtures.py
!python scripts/run_stage0_stage1.py --config configs/stage0_stage1_fixtures.yaml --print-summary

## Real-data Stage 1

Run the next cell **only after** UCM-Captions and RSITMD exist at the Drive paths above.
`--print-paths` should show `exists: true` for `ucm_root` and `rsitmd_root`.

In [ ]:
# 5) Real UCM + RSITMD leakage audit (Stage 1 only). Uncomment when data is ready.
# !python scripts/run_stage0_stage1.py --config configs/stage0_stage1_ucm_rsitmd.yaml --print-summary

## Stage 2 — retrieval only

Run this section only after the real Stage 1 cell passes. KARG-RSIC names CLIP ViT-B/32 pretrained on LAION-400M but does not publish an exact checkpoint ID. R0 records OpenCLIP `ViT-B-32-quickgelu` / `laion400m_e32` as our reproducible interpretation.

Large embeddings, indexes, ranked results, and reports are written to Drive. No caption generator or training is included.

In [ ]:
# 6) Install retrieval-only dependencies and run R0.
!pip install -q -r requirements-stage2.txt

import os
os.environ["RSRAG_UCM_MANIFEST"] = "/content/drive/MyDrive/rsrag-artifacts/processed/ucm_captions_manifest.jsonl"
os.environ["RSRAG_RSITMD_MANIFEST"] = "/content/drive/MyDrive/rsrag-artifacts/processed/rsitmd_manifest.jsonl"
os.environ["RSRAG_LEAKAGE_AUDIT"] = "/content/drive/MyDrive/rsrag-artifacts/reports/leakage_audit.json"
os.environ["RSRAG_ARTIFACT_ROOT"] = "/content/drive/MyDrive/rsrag-artifacts/stage2/R0_clip_exact"

!python scripts/run_stage2_retrieval.py --config configs/stage2_retrieval.yaml --print-paths
!python scripts/run_stage2_retrieval.py --config configs/stage2_retrieval.yaml